In [2]:
import gdown
import os

url = "https://drive.google.com/drive/u/0/folders/1Oiq2ubkNc5xoN4dQDI3UYdYB1vGgYaJP"

gdown.download_folder(url, remaining_ok=True, use_cookies=False)
print("Attempted to download folder contents.")


Retrieving folder contents


Processing file 1RIgnFDnFbzMLyNLxT1AQEg7wlmNa2lZO Apple_10-Q4-2024-As-Filed.pdf
Processing file 1PrMab5gI8PcY-33-e4ishPJoqZUr1YVw Tesla_NASDAQ_TSLA_2023.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1RIgnFDnFbzMLyNLxT1AQEg7wlmNa2lZO
To: /content/ABB_LLM/Apple_10-Q4-2024-As-Filed.pdf
100%|██████████| 964k/964k [00:00<00:00, 69.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PrMab5gI8PcY-33-e4ishPJoqZUr1YVw
To: /content/ABB_LLM/Tesla_NASDAQ_TSLA_2023.pdf
100%|██████████| 985k/985k [00:00<00:00, 86.4MB/s]

Attempted to download folder contents.



Download completed


In [3]:
!pip install pymupdf
import pandas as pd
import fitz
import os
import re
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
!pip install chromadb sentence-transformers rfc3987
import chromadb
from chromadb.utils import embedding_functions
import uuid
print('ChromaDB and dependencies successfully installed and imported.')
!pip install -q sentence-transformers
from sentence_transformers import CrossEncoder
import chromadb

ChromaDB and dependencies successfully installed and imported.


In [4]:
# initialize global list to avoid duplicate entries from previous runs
structured_data = []

pdf_dir = '/content/ABB_LLM/'
file_paths = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
SECTION_HEADER_PATTERN = re.compile(r'(?i)^\s*(Item\s+\d+[A-Z]?)\b', re.MULTILINE)

def process_pdf_with_tables(file_path):
    filename = os.path.basename(file_path)
    doc = fitz.open(file_path)
    current_section = "Unknown"

    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)

        # 1. Extract standard text
        raw_text = page.get_text("text")

        # 2. Extract tables and convert to Markdown strings
        table_texts = []
        tabs = page.find_tables()
        for tab in tabs:
            df_tab = tab.to_pandas()
            if not df_tab.empty:
                # Convert table to markdown format to preserve structure
                table_texts.append(df_tab.to_markdown(index=False))

        # 3. Combine text and tables
        combined_content = raw_text
        if table_texts:
            combined_content += "\n\n### Tables Extracted:\n" + "\n\n".join(table_texts)

        # 4. Cleaning
        cleaned_text = re.sub(r'\n{2,}', '\n\n', combined_content)
        cleaned_text = re.sub(r' +', ' ', cleaned_text).strip()

        # Detect section
        section_match = SECTION_HEADER_PATTERN.search(cleaned_text)
        if section_match:
            current_section = section_match.group(1).upper()

        structured_data.append({
            'filename': filename,
            'page_number': page_num + 1,
            'section': current_section,
            'content': cleaned_text
        })
    doc.close()

# Execute processing
for path in file_paths:
   process_pdf_with_tables(path)

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

enriched_chunks = []

# 2. Iterate through structured_data to split content
for page in structured_data:
    segments = text_splitter.split_text(page['content'])
    for segment in segments:
        enriched_chunks.append({
            "text": segment,
            "metadata": {
                "filename": page['filename'],
                "page_number": page['page_number'],
                "section": page['section']
            }
        })

# 3. Verify the process
print(f"Total enriched chunks generated: {len(enriched_chunks)}")
if enriched_chunks:
    print(f"Sample chunk metadata: {enriched_chunks[0]['metadata']}")

Total enriched chunks generated: 1168
Sample chunk metadata: {'filename': 'Apple_10-Q4-2024-As-Filed.pdf', 'page_number': 1, 'section': 'Unknown'}


In [6]:
import chromadb
from chromadb.utils import embedding_functions
import uuid
print('ChromaDB and dependencies successfully installed and imported.')

ChromaDB and dependencies successfully installed and imported.


In [7]:
# 1. Initialize persistent client using the Drive path
client = chromadb.PersistentClient(path='./chroma_db_storage')

# 2. Define embedding function
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)

# 3. Delete existing collection if it exists
try:
    client.delete_collection(name='filings_appletesla')
    print("Deleted existing collection 'filings_appletesla'.")
except Exception as e:
    print(f"Collection not found or could not be deleted: {e}")

# 4. Create new collection
collection = client.create_collection(
    name='filings_appletesla',
    embedding_function=embedding_func
)

# 5. Prepare data for ingestion
documents = [chunk['text'] for chunk in enriched_chunks]
metadatas = [chunk['metadata'] for chunk in enriched_chunks]
ids = [str(uuid.uuid4()) for _ in range(len(enriched_chunks))]

# 6. Ingest data
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Successfully re-initialized collection '{collection.name}' on Drive.")
print(f"Total records stored: {collection.count()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Deleted existing collection 'filings_appletesla'.
Successfully re-initialized collection 'filings_appletesla' on Drive.
Total records stored: 1168


In [8]:
!pip install -q transformers accelerate

import transformers
import torch

print(f'Transformers version: {transformers.__version__}')

# Check for GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: No GPU detected. LLM inference will be slow on CPU.')

Transformers version: 5.0.0
Using device: cpu


In [9]:
# Define the custom RAG prompt template with strict citation enforcement
rag_prompt_template = """
Use the following pieces of retrieved context to answer the question.

--- INSTRUCTIONS ---
1. Use ONLY the information provided in the Context sources.
2. If the answer is not contained within the context (e.g. asking for future years not in the text), clearly state: \"Not Answerable\".
3. You MUST provide the answer followed by the exact citation in this format: [Filename, Section, Page X].
4. Do not write full sentences, just provide the data and the citation separating with comma.
5. If the context does not contain the specific year or data point requested, do not guess. Say \"Not Answerable\".
6. If the context does not contain specific information within the context(e.g. asking how are you) Say \"Not Answerable\" and dont provide any citation details.
7. CRITICAL: If the answer is \"Not Answerable\", do NOT provide any citation, headers, or source information.

--- CONTEXT ---
{context}

--- QUESTION ---
{query}

--- ANSWER ---
"""

print("RAG prompt template updated for strict validation.")

RAG prompt template updated for strict validation.


In [1]:
#avoid crashing of session and for memory efficient loading
!pip install -q bitsandbytes accelerate
import torch
import gc

def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

clear_memory()
print("Memory cleared and optimization libraries installed.")

Memory cleared and optimization libraries installed.


In [2]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoConfig

model_id = "microsoft/phi-2"
print("Loading lightweight Phi-2 model strictly on CPU to avoid CUDA OOM...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    config.pad_token_id = tokenizer.pad_token_id

    # Explicitly map to CPU and use float32
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        config=config,
        torch_dtype=torch.float32,
        device_map={ "": "cpu" },
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )

    setattr(model.config, "pad_token_id", tokenizer.pad_token_id)

    rag_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer
    )

    print("Phi-2 loaded successfully on CPU.")
except Exception as e:
    print(f"Error loading model: {e}")

def generate_rag_answer(query, ranked_chunks, top_n=1):
    context_segments = []
    for i, chunk in enumerate(ranked_chunks[:top_n]):
        source_info = f"[Source: {chunk['metadata']['filename']}, Section: {chunk['metadata']['section']}, Page: {chunk['metadata']['page_number']}]"
        context_segments.append(f"{source_info}\n{chunk['text']}")

    aggregated_context = "\n\n".join(context_segments)

    final_prompt = rag_prompt_template.format(
        context=aggregated_context,
        query=query
    )

    sequences = rag_pipeline(
        final_prompt,
        max_new_tokens=100,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.pad_token_id
    )

    return sequences[0]['generated_text']

Loading lightweight Phi-2 model strictly on CPU to avoid CUDA OOM...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Phi-2 loaded successfully on CPU.


In [3]:
!pip install -q chromadb
import numpy as np
from sentence_transformers import CrossEncoder
import chromadb
from chromadb.utils import embedding_functions

# 1. Define the test query
test_query = "What was Apples total revenue for the fiscal year ended September28,2024?"

# Ensure environment and template are ready
try:
    client
    embedding_func
except NameError:
    client = chromadb.PersistentClient(path='./chroma_db_storage')
    embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

if 'rag_prompt_template' not in globals():
    rag_prompt_template = """
Use the following pieces of retrieved context to answer the question.

--- INSTRUCTIONS ---
1. Use ONLY the information provided in the Context sources.
2. If the answer is not contained within the context (e.g. asking for future years not in the text), clearly state: \"Not Answerable\".
3. You MUST provide the answer followed by the exact citation in this format: [Filename, Section, Page X].
4. Do not write full sentences, just provide the data and the citation separating with comma.
5. If the context does not contain the specific year or data point requested, do not guess. Say \"Not Answerable\".
6. If the context does not contain specific information within the context(e.g. asking how are you) Say \"Not Answerable\" and dont provide any citation details.
7. CRITICAL: If the answer is \"Not Answerable\", do NOT provide any citation, headers, or source information.

--- CONTEXT ---
{context}

--- QUESTION ---
{query}

--- ANSWER ---
"""

# 2. Retrieval Stage
collection = client.get_collection(name='filings_appletesla', embedding_function=embedding_func)
results = collection.query(query_texts=[test_query], n_results=5)
candidates = results['documents'][0]
metadatas = results['metadatas'][0]

# 3. Re-ranking (Forcing CPU to avoid CUDA OOM)
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cpu')
pairs = [[test_query, doc] for doc in candidates]
scores = rerank_model.predict(pairs)

ranked_results = []
for i in range(len(candidates)):
    ranked_results.append({
        "score": float(scores[i]),
        "text": candidates[i],
        "metadata": metadatas[i]
    })
ranked_results = sorted(ranked_results, key=lambda x: x['score'], reverse=True)

# 4. Generation Stage
print(f"--- Generating Answer for: {test_query} ---\n")
answer = generate_rag_answer(test_query, ranked_results, top_n=1)

# Refined post-processing
clean_answer = answer.strip()
if "Not Answerable" in clean_answer:
    clean_answer = "Not Answerable"
else:
    lines = [l.strip() for l in clean_answer.split('\n') if l.strip()]
    result_lines = []
    for line in lines:
        result_lines.append(line)
        if "]" in line and "[" in line and (".pdf" in line.lower() or "Page" in line):
            break
    clean_answer = "\n".join(result_lines)

print(clean_answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_g

--- Generating Answer for: What was Apples total revenue for the fiscal year ended September28,2024? ---

$391,035
--- CITATION ---
[Source: Apple_10-Q4-2024-As-Filed.pdf, Section: ITEM 8, Page: 38]
